In [1]:
import pandas as pd # Principal biblioteca utilizada para manipular tabelas

tabela2017 = pd.read_csv("datatran2017.csv", encoding="latin-1", sep=";")
tabela2018 = pd.read_csv("datatran2018.csv", encoding="latin-1", sep=";")
tabela2019 = pd.read_csv("datatran2019.csv", encoding="latin-1", sep=";")
tabela2020 = pd.read_csv("datatran2020.csv", encoding="latin-1", sep=";")
tabela2021 = pd.read_csv("datatran2021.csv", encoding="latin-1", sep=";")
tabela2022 = pd.read_csv("datatran2022.csv", encoding="latin-1", sep=";")
tabela2023 = pd.read_csv("datatran2023.csv", encoding="latin-1", sep=";")
tabela2024 = pd.read_csv("datatran2024.csv", encoding="latin-1", sep=";")
tabela2025 = pd.read_csv("datatran2025.csv", encoding="latin-1", sep=";")
# Tabelas de 2017 a 2025 lidas

tabela_completa = pd.concat([tabela2017, tabela2018, tabela2019, tabela2020, tabela2021, tabela2022, tabela2023, tabela2024, tabela2025], ignore_index=True) # Concatena todas as tabelas em uma só
tabela_completa.to_csv("tabela_completa.csv", index=False, encoding="latin-1", sep=";") # Transforma a tabela concatenada em um arquivo .csv
display(tabela_completa) # Visualização da tabela completa


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,17.0,2017-01-01,domingo,01:45:00,RS,116,"34,9",VACARIA,Defeito Mecânico no Veículo,Colisão traseira,...,0,2,0,4,2,"-28,5071196","-50,941176",SPRF-RS,DEL05-RS,UOP03-DEL05-RS
1,20.0,2017-01-01,domingo,01:00:00,PR,376,636,TIJUCAS DO SUL,Velocidade Incompatível,Saída de leito carroçável,...,0,1,0,0,2,"-25,754","-49,1266",SPRF-PR,DEL01-PR,DEL7/1-UOP08/PR
2,69.0,2017-01-01,domingo,04:40:00,BA,101,65,ENTRE RIOS,Condutor Dormindo,Colisão frontal,...,1,2,0,2,2,"-11,9618","-38,0953",SPRF-BA,DEL01-BA,UOP04-DEL01-BA
3,106.0,2017-01-01,domingo,06:30:00,PA,316,"72,5",CASTANHAL,Falta de Atenção à Condução,Colisão lateral,...,0,3,0,0,3,"-1,2899799","-47,83483207",SPRF-PA,DEL01-PA,DEL19/1-UOP02/PA
4,109.0,2017-01-01,domingo,09:00:00,GO,20,"220,5",POSSE,Defeito na Via,Colisão com objeto estático,...,1,0,1,3,2,"-14,14220931","-46,32258922",SPRF-DF,DEL02-DF,UOP03-DEL02-DF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632662,757462.0,2025-12-20,sábado,13:10:00,MG,381,"279,5",JAGUARACU,Acessar a via sem observar a presença dos outr...,Colisão transversal,...,0,1,0,2,2,"-19,607","-42,7611",SPRF-MG,DEL03-MG,UOP02-DEL03-MG
632663,757492.0,2025-09-27,sábado,13:50:00,SC,282,381,HERVAL DOESTE,Conversão proibida,Colisão transversal,...,1,0,2,2,3,"-27,2096","-51,50077",SPRF-SC,DEL07-SC,UOP05-DEL07-SC
632664,757593.0,2025-12-14,domingo,13:50:00,PE,104,62,CARUARU,Ausência de reação do condutor,Colisão transversal,...,0,2,0,1,2,"-8,25368","-35,97546",SPRF-PE,DEL02-PE,UOP01-DEL02-PE
632665,758175.0,2025-12-15,segunda-feira,15:50:00,SC,101,130,CAMBORIU,Condutor deixou de manter distância do veículo...,Colisão traseira,...,0,2,0,0,2,"-26,99305955","-48,6609548",SPRF-SC,DEL04-SC,UOP03-DEL04-SC


In [2]:
tabela_limpa = tabela_completa.drop(columns=["id","horario","uf","br","km","municipio",
"fase_dia","sentido_via","tipo_pista","uso_solo","latitude",
"longitude","regional","delegacia","uop"]) # "Excluímos" todas as colunas que NÃO utilizaremos no projeto

tabela_limpa["causa_acidente"] = (
    tabela_limpa["causa_acidente"]
    .str.strip()
    .str.lower()
) # Transforma a coluna 'causa_acidente' para minúsculo e exclui espaços sobressalentes para melhor manuseio da coluna especificada

tabela_limpa = tabela_limpa.dropna() # Exclui todas as linhas que possuir alguma coluna vazia

tabela_limpa["ano"] = pd.to_datetime(
    tabela_limpa["data_inversa"]
).dt.year # Cria uma nova coluna 'ano' a partir da coluna 'data_inversa' (que possui o formato de data) e extrai apenas o ano para facilitar a análise dos dados por ano

tabela_limpa.drop(columns=["data_inversa"], inplace=True) # Exclui a coluna 'data_inversa' (que possui o formato de data) pois já extraímos o ano para a nova coluna 'ano' e não precisamos mais da coluna original

# A seguir, transformamos as causas de acidentes que poderiam ser resumidos por categorias e subcategorias
# para facilitar o uso dos dados futuramente.

estrutura_causa_cat = {
    "problemas_mecanicos": [
        'defeito mecânico no veículo',
        'demais falhas mecânicas ou elétricas',
        'problema com o freio',
        'problema na suspensão',
        'faróis desregulados',
        'avarias e/ou desgaste excessivo no pneu',
        'deficiência ou não acionamento do sistema de iluminação/sinalização do veículo'
    ],

    "problemas_na_via": [
        'ausência de sinalização',
        'acostamento em desnível',
        'acumulo de areia ou detritos sobre o pavimento',
        'acumulo de água sobre o pavimento',
        'acumulo de óleo sobre o pavimento',
        'afundamento ou ondulação no pavimento',
        'animais na pista',
        'curva acentuada',
        'declive acentuado',
        'defeito na via',
        'deficiência do sistema de iluminação/sinalização',
        'demais falhas na via',
        'desvio temporário',
        'faixas de trânsito com largura insuficiente',
        'falta de acostamento',
        'falta de elemento de contenção que evite a saída do leito carroçável',
        'iluminação deficiente',
        'objeto estático sobre o leito carroçável',
        'obras na pista',
        'obstrução na via',
        'pista em desnível',
        'pista esburacada',
        'pista escorregadia',
        'redutor de velocidade em desacordo',
        'semáforo com defeito',
        'sinalização da via insuficiente ou inadequada',
        'sinalização encoberta',
        'sinalização mal posicionada',
        'sistema de drenagem ineficiente',
        'área urbana sem a presença de local apropriado para a travessia de pedestres',
        'restrição de visibilidade',
        'restrição de visibilidade em curvas horizontais',
        'restrição de visibilidade em curvas verticais'
    ],

    "fatores_criminais": [
        'agressão externa',
        'obstrução via tentativa assalto'
    ],

    "fenomenos_natureza": [
        'chuva',
        'demais fenômenos da natureza',
        'fenômenos da natureza',
        'fumaça',
        'neblina',
    ],

    "falha_humana": [
        'ausência de reação do condutor',
        'carga excessiva e/ou mal acondicionada',
        'condutor dormindo',
        'frear bruscamente',
        'reação tardia ou ineficiente do condutor'
    ],

    "alcool_ou_drogas": [
        'ingestão de substâncias psicoativas pelo condutor',
        'ingestão de álcool pelo condutor',
        'ingestão de álcool e/ou substâncias psicoativas pelo pedestre',
        'ingestão de álcool ou de substâncias psicoativas pelo pedestre',
        'pedestre - ingestão de álcool/ substâncias psicoativas',
        'ingestão de substâncias psicoativas',
        'ingestão de álcool'
    ],

    "problemas_saude": [
        'mal súbito do condutor',
        'mal súbito',
        'transtornos mentais (exceto suicidio)',
        'suicídio (presumido)'
    ],

    "infracoes": [
        'acessar a via sem observar a presença dos outros veículos',
        'acesso irregular',
        'condutor deixou de manter distância do veículo da frente',
        'condutor desrespeitou a iluminação vermelha do semáforo',
        'condutor usando celular',
        'conversão proibida',
        'deixar de acionar o farol da motocicleta (ou similar)',
        'desobediência às normas de trânsito pelo condutor',
        'desrespeitar a preferência no cruzamento',
        'estacionar ou parar em local proibido',
        'falta de atenção à condução',
        'manobra de mudança de faixa',
        'modificação proibida',
        'não guardar distância de segurança',
        'participar de racha',
        'retorno proibido',
        'trafegar com motocicleta (ou similar) entre as faixas',
        'transitar na calçada',
        'transitar na contramão',
        'transitar no acostamento',
        'ultrapassagem indevida',
        'velocidade incompatível',
        'desobediência às normas de trânsito pelo pedestre',
        'entrada inopinada do pedestre',
        'falta de atenção do pedestre',
        'pedestre andava na pista',
        'pedestre cruzava a pista fora da faixa'
    ],
}

estrutura_causa_sub = {
    "alcool_ou_drogas": {
        "condutor": [
            'ingestão de substâncias psicoativas pelo condutor',
            'ingestão de álcool pelo condutor'
        ],
        "pedestre": [
            'ingestão de álcool e/ou substâncias psicoativas pelo pedestre',
            'ingestão de álcool ou de substâncias psicoativas pelo pedestre',
            'pedestre - ingestão de álcool/ substâncias psicoativas'
        ],
        "indefinido": [
            'ingestão de substâncias psicoativas',
            'ingestão de álcool'
        ]
    },

    "problemas_saude": {
        "condutor": [
            'mal súbito do condutor'
        ],
        "indefinido": [
            'mal súbito',
            'transtornos mentais (exceto suicidio)',
            'suicídio (presumido)'
        ]
    },

    "infracoes": {
        "condutor": [
            'acessar a via sem observar a presença dos outros veículos',
            'acesso irregular',
            'condutor deixou de manter distância do veículo da frente',
            'condutor desrespeitou a iluminação vermelha do semáforo',
            'condutor usando celular',
            'conversão proibida',
            'deixar de acionar o farol da motocicleta (ou similar)',
            'desobediência às normas de trânsito pelo condutor',
            'desrespeitar a preferência no cruzamento',
            'estacionar ou parar em local proibido',
            'falta de atenção à condução',
            'manobra de mudança de faixa',
            'modificação proibida',
            'não guardar distância de segurança',
            'participar de racha',
            'retorno proibido',
            'trafegar com motocicleta (ou similar) entre as faixas',
            'transitar na calçada',
            'transitar na contramão',
            'transitar no acostamento',
            'ultrapassagem indevida',
            'velocidade incompatível',
        ],
        "pedestre": [
            'desobediência às normas de trânsito pelo pedestre',
            'entrada inopinada do pedestre',
            'falta de atenção do pedestre',
            'pedestre andava na pista',
            'pedestre cruzava a pista fora da faixa',
        ]
    }
}

# Início

mapa_causa_cat = {}
mapa_causa_sub = {}

for cat_causa, causas in estrutura_causa_cat.items():
    for causa in causas:
        mapa_causa_cat[causa] = cat_causa

tabela_limpa["cat_causa"] = (
    tabela_limpa["causa_acidente"]
    .map(mapa_causa_cat)
)

for cat_causa, subcats in estrutura_causa_sub.items():
    for sub_causa, causas in subcats.items():
        for causa in causas:
            mapa_causa_sub[causa] = sub_causa

tabela_limpa["sub_causa"] = (
    tabela_limpa["causa_acidente"]
    .map(mapa_causa_sub)
    .fillna("outro")
)

# Do 'Início' até esse ponto, o algoritimo leu a coluna 'causa_acidente' e criou duas novas colunas (cat_causa e sub_causa) com os valores "substituídos".

# A seguir, fazemos a mesma coisa para 'tipo_acidente', resumindo os dados semelhantes em uma categoria
# para facilitar o uso dos dados futuramente.

tabela_limpa["tipo_acidente"] = (
    tabela_limpa["tipo_acidente"]
    .str.strip()
    .str.lower()
)

estrutura_tipo_cat = {
    "colisao": [
        'colisão com objeto',
        'colisão com objeto em movimento',
        'colisão com objeto estático',
        'colisão frontal',
        'colisão lateral',
        'colisão lateral mesmo sentido',
        'colisão lateral sentido oposto',
        'colisão transversal',
        'colisão traseira',
    ]
}

# Início

mapa_tipo_cat = {}

for cat_tipo, tipos in estrutura_tipo_cat.items():
    for tipo in tipos:
        mapa_tipo_cat[tipo] = cat_tipo

tabela_limpa["tipo_acidente"] = (
    tabela_limpa["tipo_acidente"]
    .replace(mapa_tipo_cat)
)

# Do 'Início' até esse ponto, o algoritimo leu a coluna 'tipo_acidente' e, onde necessário, substituiu os valores pela categoria criada anteriormente

display(tabela_limpa) # Impressão da tabela após as modificações
display(tabela_limpa.info()) # Impressãp dos tipos de cada coluna


,dia_semana,causa_acidente,tipo_acidente,classificacao_acidente,condicao_metereologica,tracado_via,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,ano,cat_causa,sub_causa
0,domingo,defeito mecânico no veículo,colisao,Com Vítimas Feridas,Céu Claro,Reta,6,0,4,0,2,0,4,2,2017,problemas_mecanicos,outro
1,domingo,velocidade incompatível,saída de leito carroçável,Com Vítimas Fatais,Garoa/Chuvisco,Curva,2,1,0,0,1,0,0,2,2017,infracoes,condutor
2,domingo,condutor dormindo,colisao,Com Vítimas Fatais,Nublado,Curva,5,1,1,1,2,0,2,2,2017,falha_humana,outro
3,domingo,falta de atenção à condução,colisao,Com Vítimas Fatais,Céu Claro,Reta,4,1,0,0,3,0,0,3,2017,infracoes,condutor
5,domingo,ingestão de álcool,colisao,Com Vítimas Fatais,Céu Claro,Reta,3,1,0,0,1,3,0,5,2017,alcool_ou_drogas,indefinido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632662,sábado,acessar a via sem observar a presença dos outr...,colisao,Com Vítimas Feridas,Céu Claro,Interseção de Vias,3,0,2,0,1,0,2,2,2025,infracoes,condutor
632663,sábado,conversão proibida,colisao,Com Vítimas Feridas,Céu Claro,Reta;Declive,4,0,1,1,0,2,2,3,2025,infracoes,condutor
632664,domingo,ausência de reação do condutor,colisao,Com Vítimas Feridas,Céu Claro,Reta,3,0,1,0,2,0,1,2,2025,falha_humana,outro
632665,segunda-feira,condutor deixou de manter distância do veículo...,colisao,Sem Vítimas,Sol,Reta,2,0,0,0,2,0,0,2,2025,infracoes,condutor


<class 'pandas.core.frame.DataFrame'>
Index: 632617 entries, 0 to 632666
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   dia_semana              632617 non-null  object
 1   causa_acidente          632617 non-null  object
 2   tipo_acidente           632617 non-null  object
 3   classificacao_acidente  632617 non-null  object
 4   condicao_metereologica  632617 non-null  object
 5   tracado_via             632617 non-null  object
 6   pessoas                 632617 non-null  int64 
 7   mortos                  632617 non-null  int64 
 8   feridos_leves           632617 non-null  int64 
 9   feridos_graves          632617 non-null  int64 
 10  ilesos                  632617 non-null  int64 
 11  ignorados               632617 non-null  int64 
 12  feridos                 632617 non-null  int64 
 13  veiculos                632617 non-null  int64 
 14  ano                     632617 non-null  

None

In [3]:
from sklearn.preprocessing import LabelEncoder # Importando da biblioteca 'sklearn' o 'LabelEncoder' que utilizaremos para transformar as colunas de tipo 'object' em 'int64' (dados são numerados automaticamente)

dia_semana_encoder = LabelEncoder()
# tabela_limpa["dia_semana"] = dia_semana_encoder.fit_transform(tabela_limpa["dia_semana"])

tipo_acidente_encoder = LabelEncoder()
# tabela_limpa["tipo_acidente"] = tipo_acidente_encoder.fit_transform(tabela_limpa["tipo_acidente"])

classificacao_acidente_encoder = LabelEncoder()
# tabela_limpa["classificacao_acidente"] = classificacao_acidente_encoder.fit_transform(tabela_limpa["classificacao_acidente"])

condicao_metereologica_encoder = LabelEncoder()
# tabela_limpa["condicao_metereologica"] = condicao_metereologica_encoder.fit_transform(tabela_limpa["condicao_metereologica"])

cat_causa_encoder = LabelEncoder()
# tabela_limpa["cat_causa"] = cat_causa_encoder.fit_transform(tabela_limpa["cat_causa"])

sub_causa_encoder = LabelEncoder()
# tabela_limpa["sub_causa"] = sub_causa_encoder.fit_transform(tabela_limpa["sub_causa"])

# Nas linhas acima, fizemos 1 encoder para cada coluna que precisamos transformar em 'int64' e substituímos na tabela (descontinuado para melhor visualização da tabela)

In [12]:
import plotly.express as px

# Agrupa por ano
resultado = (
    tabela_limpa
    .groupby("ano")
    .agg({
        "mortos": "sum",
        "feridos_graves": "sum"
    })
    .reset_index()
)

# Adiciona total de acidentes
resultado["total_acidentes"] = (
    tabela_limpa
    .groupby("ano")
    .size()
    .values
)

# Reorganiza colunas
resultado = resultado[
    ["ano", "total_acidentes", "mortos", "feridos_graves"]
]

# Cria gráfico
fig = px.line(
    resultado,
    x="ano",
    y=["total_acidentes", "mortos", "feridos_graves"],
    markers=True,
    title="Evolução dos acidentes, mortes e feridos graves por ano",
    labels={
        "value": "Quantidade",
        "ano": "Ano",
        "variable": "Indicador"
    }
)

fig.update_layout(
    hovermode="x unified"
)

fig.show()

In [ ]:
# ============================================
# 1. Criar coluna de gravidade
# ============================================

tabela_limpa["grave_ou_obito"] = (
    (tabela_limpa["mortos"] > 0) |
    (tabela_limpa["feridos_graves"] > 0)
)

# ============================================
# 2. GRÁFICO 1 - IMPACTO TOTAL
# ============================================
# Mostra:
# Quais categorias geram mais vítimas
# graves e mortes no total
# ============================================

resultado_total = (
    tabela_limpa
    .groupby("cat_causa")[["mortos", "feridos_graves"]]
    .sum()
    .reset_index()
)

resultado_total["graves_ou_mortos"] = (
    resultado_total["mortos"] +
    resultado_total["feridos_graves"]
)

resultado_total = resultado_total.sort_values(
    by="graves_ou_mortos",
    ascending=False
)

fig1 = px.bar(
    resultado_total,
    x="graves_ou_mortos",
    y="cat_causa",
    orientation="h",
    title="Total de lesões graves e óbitos por categoria",
    labels={
        "graves_ou_mortos": "Total",
        "cat_causa": "Categoria"
    }
)

fig1.update_layout(
    yaxis={'categoryorder':'total ascending'}
)

fig1.show()

# ============================================
# 3. GRÁFICO 2 - TAXA DE GRAVIDADE
# ============================================
# Mostra:
# Quando a categoria acontece,
# qual a chance de gerar morte
# ou ferido grave
# ============================================

resultado_taxa = (
    tabela_limpa
    .groupby("cat_causa")
    .agg({
        "grave_ou_obito": "mean",
        "causa_acidente": "count"
    })
    .rename(columns={
        "grave_ou_obito": "taxa_gravidade",
        "causa_acidente": "quantidade_acidentes"
    })
    .reset_index()
)

# ============================================
# 4. Filtrar categorias pequenas
# ============================================
# Evita distorções estatísticas
# ============================================

resultado_taxa = resultado_taxa[
    resultado_taxa["quantidade_acidentes"] >= 100
]

# ============================================
# 5. Ordenar
# ============================================

resultado_taxa = resultado_taxa.sort_values(
    by="taxa_gravidade",
    ascending=False
)

# ============================================
# 6. Criar gráfico da taxa
# ============================================

fig2 = px.bar(
    resultado_taxa,
    x="taxa_gravidade",
    y="cat_causa",
    orientation="h",
    title="Taxa de acidentes com lesões graves ou óbitos",
    labels={
        "taxa_gravidade": "Taxa",
        "cat_causa": "Categoria"
    }
)

fig2.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_tickformat=".0%"
)

fig2.show()

In [5]:
# ============================================
# 1. Limpar tracado_via
# ============================================

tabela_limpa["tracado_via"] = (
    tabela_limpa["tracado_via"]
    .str.split(";")
    .str[0]
    .str.strip()
    .str.lower()
)

# ============================================
# 2. Selecionar top tipos
# ============================================

top_tipos = (
    tabela_limpa["tipo_acidente"]
    .value_counts()
    .head(10)
    .index
)

tabela_top = tabela_limpa[
    tabela_limpa["tipo_acidente"].isin(top_tipos)
]

# ============================================
# 3. Criar tabela percentual
# ============================================

relacao_pct = pd.crosstab(
    tabela_top["tracado_via"],
    tabela_top["tipo_acidente"],
    normalize="index"
) * 100

relacao_pct = relacao_pct.round(1)

# ============================================
# 4. HEATMAP
# ============================================

fig_heatmap = px.imshow(
    relacao_pct,
    text_auto=True,
    aspect="auto",
    title="Heatmap - Relação entre traçado da via e tipo de acidente",
    labels={
        "x": "Tipo de acidente",
        "y": "Traçado da via",
        "color": "%"
    }
)

fig_heatmap.update_layout(
    width=1200,
    height=700
)

fig_heatmap.show()

In [6]:
# =====================================================
# 1. Criar coluna indicando se houve óbito
# =====================================================

tabela_limpa["houve_obito"] = tabela_limpa["mortos"] > 0

# =====================================================
# 2. Agrupar combinações
# =====================================================

resultado = (
    tabela_limpa
    .groupby([
        "cat_causa",
        "tipo_acidente",
        "tracado_via"
    ])
    .agg(
        total_acidentes=("mortos", "size"),
        total_obitos=("houve_obito", "sum"),
        media_mortos=("mortos", "mean"),
        media_feridos_graves=("feridos_graves", "mean")
    )
    .reset_index()
)

# =====================================================
# 3. Calcular taxa de óbito
# =====================================================

resultado["taxa_obito"] = (
    resultado["total_obitos"]
    / resultado["total_acidentes"]
) * 100

# =====================================================
# 4. Filtrar grupos pequenos
# =====================================================

resultado = resultado[
    resultado["total_acidentes"] >= 100
]

# =====================================================
# 5. Criar coluna com combinação
# =====================================================

resultado["combinacao"] = (
    resultado["cat_causa"] + " | " +
    resultado["tipo_acidente"] + " | " +
    resultado["tracado_via"]
)

# =====================================================
# 6. TOP 10 - TAXA DE ÓBITO
# =====================================================

top_taxa = (
    resultado
    .sort_values(by="taxa_obito", ascending=False)
    .head(10)
)

fig1 = px.bar(
    top_taxa,
    x="taxa_obito",
    y="combinacao",
    orientation="h",
    title="Top 10 combinações com maior taxa de óbito",
    labels={
        "taxa_obito": "Taxa de óbito (%)",
        "combinacao": "Combinação"
    },
    text="taxa_obito"
)

fig1.update_traces(texttemplate='%{text:.1f}%')

fig1.update_layout(
    yaxis={'categoryorder':'total ascending'}
)

fig1.show()

# =====================================================
# 7. TOP 10 - MÉDIA DE MORTOS
# =====================================================

top_mortos = (
    resultado
    .sort_values(by="media_mortos", ascending=False)
    .head(10)
)

fig2 = px.bar(
    top_mortos,
    x="media_mortos",
    y="combinacao",
    orientation="h",
    title="Top 10 combinações com maior média de mortos",
    labels={
        "media_mortos": "Média de mortos",
        "combinacao": "Combinação"
    },
    text="media_mortos"
)

fig2.update_traces(texttemplate='%{text:.1f}')

fig2.update_layout(
    yaxis={'categoryorder':'total ascending'}
)

fig2.show()

# =====================================================
# 8. TOP 10 - MÉDIA DE FERIDOS GRAVES
# =====================================================

top_feridos = (
    resultado
    .sort_values(by="media_feridos_graves", ascending=False)
    .head(10)
)

fig3 = px.bar(
    top_feridos,
    x="media_feridos_graves",
    y="combinacao",
    orientation="h",
    title="Top 10 combinações com maior média de feridos graves",
    labels={
        "media_feridos_graves": "Média de feridos graves",
        "combinacao": "Combinação"
    },
    text="media_feridos_graves"
)

fig3.update_traces(texttemplate='%{text:.1f}')

fig3.update_layout(
    yaxis={'categoryorder':'total ascending'}
)

fig3.show()

In [7]:
# ============================================
# 1. Filtrar acidentes relacionados a álcool
# ============================================

alcool = tabela_limpa[
    tabela_limpa["cat_causa"] == "alcool_ou_drogas"
]

# ============================================
# 2. Contar tipos de acidente
# ============================================

resultado = (
    alcool["tipo_acidente"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)

resultado.columns = [
    "tipo_acidente",
    "porcentagem"
]

# ============================================
# 3. Pegar TOP 10
# ============================================

resultado = resultado.head(10)

# ============================================
# 4. Gráfico
# ============================================

fig = px.bar(
    resultado,
    x="porcentagem",
    y="tipo_acidente",
    orientation="h",
    title="Tipos de acidente mais associados à ingestão de álcool/drogas",
    labels={
        "porcentagem": "Porcentagem (%)",
        "tipo_acidente": "Tipo de acidente"
    },
    text="porcentagem"
)

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.update_layout(
    width=1200,
    height=700,
    yaxis={'categoryorder':'total ascending'}
)

fig.show()

In [8]:
import pandas as pd
import plotly.express as px

# =====================================================
# 1. Criar coluna alvo
# =====================================================

tabela_limpa["capotamento_ou_tombamento"] = (
    tabela_limpa["tipo_acidente"].isin([
        "capotamento",
        "tombamento"
    ])
)

# =====================================================
# 2. Agrupar combinações
# =====================================================

resultado = (
    tabela_limpa
    .groupby([
        "cat_causa",
        "tracado_via",
        "condicao_metereologica",
    ])
    .agg(
        total_acidentes=("tipo_acidente", "size"),
        total_capotamentos=("capotamento_ou_tombamento", "sum")
    )
    .reset_index()
)

# =====================================================
# 3. Calcular taxa
# =====================================================

resultado["taxa_capotamento"] = (
    resultado["total_capotamentos"]
    / resultado["total_acidentes"]
) * 100

# =====================================================
# 4. Filtrar grupos pequenos
# =====================================================

resultado = resultado[
    resultado["total_acidentes"] >= 100
]

# =====================================================
# 5. Criar rótulo
# =====================================================

resultado["combinacao"] = (
    resultado["cat_causa"].astype(str) + " | " +
    resultado["tracado_via"].astype(str) + " | " +
    resultado["condicao_metereologica"].astype(str)
)

# =====================================================
# 6. TOP 10
# =====================================================

top10 = (
    resultado
    .sort_values(by="taxa_capotamento", ascending=False)
    .head(10)
)

display(top10)

# =====================================================
# 7. Gráfico
# =====================================================

fig = px.bar(
    top10,
    x="taxa_capotamento",
    y="combinacao",
    orientation="h",
    title="Combinações com maior risco de capotamento/tombamento",
    labels={
        "taxa_capotamento": "Taxa (%)",
        "combinacao": "Combinação"
    },
    text="taxa_capotamento"
)

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.update_layout(
    width=1400,
    height=800,
    yaxis={'categoryorder':'total ascending'}
)

fig.show()

,cat_causa,tracado_via,condicao_metereologica,total_acidentes,total_capotamentos,taxa_capotamento,combinacao
572,problemas_na_via,rotatória,Nublado,117,59,50.427350,problemas_na_via | rotatória | Nublado
568,problemas_na_via,rotatória,Céu Claro,395,158,40.000000,problemas_na_via | rotatória | Céu Claro
518,problemas_na_via,declive,Sol,136,49,36.029412,problemas_na_via | declive | Sol
509,problemas_na_via,curva,Sol,425,151,35.529412,problemas_na_via | curva | Sol
561,problemas_na_via,retorno regulamentado,Céu Claro,247,84,34.008097,problemas_na_via | retorno regulamentado | Céu...
425,problemas_mecanicos,curva,Sol,533,176,33.020638,problemas_mecanicos | curva | Sol
521,problemas_na_via,desvio temporário,Céu Claro,134,43,32.089552,problemas_na_via | desvio temporário | Céu Claro
580,problemas_na_via,viaduto,Céu Claro,245,76,31.020408,problemas_na_via | viaduto | Céu Claro
567,problemas_na_via,rotatória,Chuva,126,38,30.158730,problemas_na_via | rotatória | Chuva
419,problemas_mecanicos,curva,Céu Claro,3927,1171,29.819200,problemas_mecanicos | curva | Céu Claro


In [26]:
import pandas as pd
import plotly.express as px

# =====================================================
# OUTLIERS - MORTOS
# =====================================================

Q1_mortos = tabela_limpa["mortos"].quantile(0.25)
Q3_mortos = tabela_limpa["mortos"].quantile(0.75)

IQR_mortos = Q3_mortos - Q1_mortos

limite_superior_mortos = Q3_mortos + (1.5 * IQR_mortos)

outliers_mortos = tabela_limpa[
    tabela_limpa["mortos"] > limite_superior_mortos
]

print("Limite superior (mortos):", limite_superior_mortos)
print("Quantidade de outliers:", len(outliers_mortos))

display(
    outliers_mortos
    .sort_values(by="mortos", ascending=False)
    .head(20)
)

# =====================================================
# BOXPLOT - MORTOS
# =====================================================

fig1 = px.box(
    tabela_limpa,
    y="mortos",
    title="Boxplot - Quantidade de mortos"
)

fig1.show()

# =====================================================
# OUTLIERS - VEÍCULOS
# =====================================================

Q1_veiculos = tabela_limpa["veiculos"].quantile(0.25)
Q3_veiculos = tabela_limpa["veiculos"].quantile(0.75)

IQR_veiculos = Q3_veiculos - Q1_veiculos

limite_superior_veiculos = Q3_veiculos + (1.5 * IQR_veiculos)

outliers_veiculos = tabela_limpa[
    tabela_limpa["veiculos"] > limite_superior_veiculos
]

print("Limite superior (veículos):", limite_superior_veiculos)
print("Quantidade de outliers:", len(outliers_veiculos))

display(
    outliers_veiculos
    .sort_values(by="veiculos", ascending=False)
    .head(20)
)

# =====================================================
# BOXPLOT - VEÍCULOS
# =====================================================

fig2 = px.box(
    tabela_limpa,
    y="veiculos",
    title="Boxplot - Quantidade de veículos"
)

fig2.show()

Limite superior (mortos): 0.0
Quantidade de outliers: 43431


,dia_semana,causa_acidente,tipo_acidente,classificacao_acidente,condicao_metereologica,tracado_via,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,ano,cat_causa,sub_causa,grave_ou_obito,houve_obito,capotamento_ou_tombamento
505488,sábado,carga excessiva e/ou mal acondicionada,colisao,Com Vítimas Fatais,Céu Claro,declive,54,37,8,5,3,2,13,7,2024,falha_humana,outro,True,True,False
487906,domingo,transitar na contramão,colisao,Com Vítimas Fatais,Céu Claro,reta,32,23,0,9,0,0,9,2,2024,infracoes,condutor,True,True,False
11359,quinta-feira,velocidade incompatível,tombamento,Com Vítimas Fatais,Nublado,curva,44,21,13,9,0,1,22,5,2017,infracoes,condutor,True,True,True
319848,segunda-feira,problema com o freio,colisao,Com Vítimas Fatais,Nublado,curva,54,19,25,8,1,1,33,2,2021,problemas_mecanicos,outro,True,True,False
624041,sexta-feira,velocidade incompatível,saída de leito carroçável,Com Vítimas Fatais,Céu Claro,declive,37,16,2,19,0,0,21,1,2025,infracoes,condutor,True,True,False
505019,domingo,demais falhas na via,queda de ocupante de veículo,Com Vítimas Fatais,Céu Claro,ponte,30,14,0,1,10,12,1,24,2024,problemas_na_via,outro,True,True,False
285262,sexta-feira,defeito mecânico no veículo,colisao,Com Vítimas Fatais,Céu Claro,aclive,50,13,13,19,4,1,32,2,2020,problemas_mecanicos,outro,True,True,False
90779,sábado,condutor dormindo,colisao,Com Vítimas Fatais,Céu Claro,reta,81,13,33,6,28,8,39,14,2018,falha_humana,outro,True,True,False
307449,quarta-feira,velocidade incompatível,tombamento,Com Vítimas Fatais,Céu Claro,curva,32,12,5,12,2,2,17,5,2021,infracoes,condutor,True,True,True
237876,domingo,objeto estático sobre o leito carroçável,colisao,Com Vítimas Fatais,Céu Claro,reta,14,12,0,1,0,1,1,3,2020,problemas_na_via,outro,True,True,False


Limite superior (veículos): 3.5
Quantidade de outliers: 43789


,dia_semana,causa_acidente,tipo_acidente,classificacao_acidente,condicao_metereologica,tracado_via,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,ano,cat_causa,sub_causa,grave_ou_obito,houve_obito,capotamento_ou_tombamento
436774,sábado,neblina,colisao,Com Vítimas Fatais,Nevoeiro/Neblina,declive,95,5,6,6,38,88,12,131,2023,fenomenos_natureza,outro,True,True,False
570398,segunda-feira,demais falhas mecânicas ou elétricas,incêndio,Sem Vítimas,Céu Claro,reta,2,0,0,0,1,81,0,82,2025,problemas_mecanicos,outro,False,False,False
171924,terça-feira,velocidade incompatível,saída de leito carroçável,Com Vítimas Fatais,Céu Claro,declive,3,2,0,0,0,37,0,38,2019,infracoes,condutor,True,True,False
578685,domingo,reação tardia ou ineficiente do condutor,colisao,Com Vítimas Feridas,Céu Claro,curva,33,0,4,4,24,6,8,31,2025,falha_humana,outro,True,False,False
365386,domingo,ultrapassagem indevida,colisao,Com Vítimas Fatais,Céu Claro,reta,5,1,1,0,2,24,1,28,2022,infracoes,condutor,True,True,False
501068,quinta-feira,neblina,engavetamento,Com Vítimas Fatais,Nublado,curva,19,1,1,6,10,12,7,26,2024,fenomenos_natureza,outro,True,True,False
431562,segunda-feira,velocidade incompatível,engavetamento,Com Vítimas Fatais,Céu Claro,reta,29,3,11,7,6,8,18,25,2023,infracoes,condutor,True,True,False
20848,terça-feira,restrição de visibilidade,engavetamento,Com Vítimas Fatais,Nevoeiro/Neblina,declive,23,4,5,0,13,10,5,25,2017,problemas_na_via,outro,True,True,False
239897,domingo,restrição de visibilidade,colisao,Com Vítimas Fatais,Nevoeiro/Neblina,reta,44,7,15,4,13,5,19,25,2020,problemas_na_via,outro,True,True,False
505019,domingo,demais falhas na via,queda de ocupante de veículo,Com Vítimas Fatais,Céu Claro,ponte,30,14,0,1,10,12,1,24,2024,problemas_na_via,outro,True,True,False
